# NatureCubePy Data Retrieval Tutorial

This notebook demonstrates NatureCubePy's data retrieval functions for accessing biodiversity observation data across multiple measurement types: camera traps (image/video), bioacoustics (audio), and environmental DNA (eDNA).

## Data Structure Overview

NatureCube projects consist of:
- **Stations**: Physical or logical measurement deployments (camera traps, audio recorders, eDNA collection sites)
- **Media Assets**: Files recorded at stations (images, video segments, audio clips)
- **Media Segments**: Time-bounded regions within an asset (e.g., video clip 0:30-0:45, species present in audio segment)
- **Labels**: Species identifications or other annotations on segments

All observations are geotagged with station coordinates and include measurement type and device metadata.

## API Tutorial Flow

1. **Stations**: Retrieve geolocations and metadata for all deployment sites
2. **Labels**: Access project-specific species reference data
3. **Media**: Query raw assets and their temporal segments
4. **Species Observations**: Retrieve merged records with species IDs, locations, and measurement context

Prereqs: a valid Okala API key (follow [01_authentication.ipynb](./01_authentication.ipynb) to set up) and NatureCubePy installed. 

In [25]:
import pandas as pd

from naturecubepy import (
    get_key,
    auth_headers,
    get_project,
    get_media_assets_df,
    get_project_labels_df,
    get_iucn_labels,
    get_station_info,
    plot_stations,
    get_camera_trap_data,
    get_audio_observation_data
)

ImportError: cannot import name 'get_audio_observation_data' from 'naturecubepy' (/Users/natimi/Projects/NatureCubePy/src/naturecubepy/__init__.py)

In [3]:
# Retrieve API key and set up authentication headers
api_key = get_key()
hdr = auth_headers(api_key)   

## 1. Get Project Name

In [4]:
project = get_project(hdr)
print(project)

Retrieving project data...
Received response with status code 200
Project data retrieved successfully
Setting your active project as - Tridom Crédit Biodiversité - UGF 34 & 35 Rougier
boundary=ProjectGeometryResponse(type='FeatureCollection', bbox=None, features=[MultiPolygonGeometryProject(type='Feature', geometry=MultiPolygonModel(type='MultiPolygon', bbox=None, coordinates=[[[Coordinates(lon=12.396622, lat=-0.078937, alt=None), Coordinates(lon=12.396627, lat=-0.07896, alt=None), Coordinates(lon=12.396632, lat=-0.078967, alt=None), Coordinates(lon=12.396691, lat=-0.079055, alt=None), Coordinates(lon=12.396732, lat=-0.079115, alt=None), Coordinates(lon=12.397605, lat=-0.080417, alt=None), Coordinates(lon=12.39785, lat=-0.080782, alt=None), Coordinates(lon=12.397919, lat=-0.080813, alt=None), Coordinates(lon=12.399502, lat=-0.081824, alt=None), Coordinates(lon=12.399857, lat=-0.082615, alt=None), Coordinates(lon=12.399971, lat=-0.082868, alt=None), Coordinates(lon=12.402152, lat=-0.083

## 2. Get Stations

Retrieve geolocations and metadata for all measurement stations in your project.

Stations represent the physical or logical deployment sites where data is collected. Each station has:
- **project_system_record_id**: Unique identifier for the station
- **device_id**: QR code or hardware identifier
- **geometry**: GeoJSON point or polygon (longitude, latitude)
- **measurement_type**: `"Camera"`, `"Bioacoustic"`, or `"eDNA"`

The `measurement_tpe` parameter filters stations by the data they serve:``"camera"``, ``"bioacoustic"``, or ``"eDNA"``.

In [8]:
stations = get_station_info(hdr, measurement_type='camera')
print(f"Loaded {len(stations)} stations")
stations.head()

Loaded 56 stations


,system_type,feature_id,feature_name,system_name,device_id,project_system_record_id,record_count,measurement_type,data_type,project_system_record_start_timestamp,project_system_record_end_timestamp,geometry
7,Sensor,4515,WC01,Browning Recon Force HP4,00005,5863,52,Camera,image,2025-06-23 12:10:00+00:00,2025-09-23 07:30:00+00:00,POINT (12.37271 -0.07863)
8,Sensor,4516,WC02,Browning Recon Force HP4,00030,5864,40,Camera,image,2025-06-29 11:25:00+00:00,2025-09-26 09:57:00+00:00,POINT (12.33868 0.01084)
10,Sensor,4517,WC03,Browning Recon Force HP4,00113,5866,57,Camera,image,2025-06-26 13:13:00+00:00,2025-09-24 07:55:00+00:00,POINT (12.33838 -0.07891)
11,Sensor,4518,WC04,Browning Recon Force HP4,00114,5867,433,Camera,image,2025-06-23 10:36:00+00:00,2025-09-23 09:28:00+00:00,POINT (12.3372 -0.20565)
12,Sensor,4519,WC05,Browning Recon Force HP4,00119,5868,20,Camera,image,2025-06-28 12:53:00+00:00,2025-09-25 11:03:00+00:00,POINT (12.37326 0.01247)


In [10]:
map_widget = plot_stations(stations)
map_widget

Plotting stations


## 3. Load Label Reference Data

Project labels are the species or taxa used in your study. Can specify where label was detected from (camera or bioacoustic).

In [11]:
for label_type in ["Camera", "Bioacoustic"]:
    print(f"\nProject labels for {label_type}")
    labels = get_project_labels_df(hdr, label_type)
    display(labels.head())


Project labels for Camera


,label_id,label,common_name,class_,order,family,genus,species,tags,global_labels_applied
0,51691,Agelastes niger,Black Guineafowl,Aves,Galliformes,Numididae,Agelastes,Agelastes niger,[],True
1,31545,Atherurus africanus,African Brush-tailed Porcupine,Mammalia,Rodentia,Hystricidae,Atherurus,Atherurus africanus,[],True
2,30582,Atilax paludinosus,Marsh Mongoose,Mammalia,Carnivora,Herpestidae,Atilax,Atilax paludinosus,[],True
3,106735,Aves,NaN,Aves,NaN,NaN,NaN,NaN,[],True
4,30583,Bdeogale nigripes,Black-legged Mongoose,Mammalia,Carnivora,Herpestidae,Bdeogale,Bdeogale nigripes,[],True



Project labels for Bioacoustic


,label_id,label,common_name,class_,order,family,genus,species,tags,global_labels_applied
0,44376,Abroscopus albogularis,Rufous-faced Warbler,Aves,Passeriformes,Scotocercidae,Abroscopus,Abroscopus albogularis,[],True
1,45454,Abroscopus superciliaris,Yellow-bellied Warbler,Aves,Passeriformes,Scotocercidae,Abroscopus,Abroscopus superciliaris,[],True
2,43377,Acanthagenys rufogularis,Spiny-cheeked Honeyeater,Aves,Passeriformes,Meliphagidae,Acanthagenys,Acanthagenys rufogularis,[],True
3,43406,Acanthiza lineata,Striated Thornbill,Aves,Passeriformes,Acanthizidae,Acanthiza,Acanthiza lineata,[],True
4,43399,Acanthiza pusilla,Brown Thornbill,Aves,Passeriformes,Acanthizidae,Acanthiza,Acanthiza pusilla,[],True


## 4. Media Assets & Segments

Media are the raw files (images, video, audio, DNA) and their annotations.

- **Media Assets**: Individual files with metadata (duration, size, timestamp, file path)
- **Media Segments**: Logical time windows within a file marked for analysis
  - Each segment may have 0–N labels (species identifications)
  - Includes verification status: `ai_derived`, `labeller_verified`, or `manager_verified`

In [17]:
psr_ids = stations["project_system_record_id"].dropna().astype(int).tolist()
if not psr_ids:
    raise ValueError("No project_system_record_id values found in stations")

media_assets = get_media_assets_df(hdr, "video", psr_ids)
print(f"Loaded {len(media_assets)} video media rows from PSR {psr_ids[0]}")
display(media_assets.head())

Loaded 4040 video media rows from PSR 5863


,label_id,label,common_name,class_,order,family,genus,species,tags,global_labels_applied,...,duration_in_seconds,file_size,number_of_individuals,segment_record_id,label_record_id,prediction_accuracy,manager_verified,labeller_verified,blank,segment_verification_status
0,30849,Cephalophus callipygus,Peters' Duiker,Mammalia,Artiodactyla,Bovidae,Cephalophus,Cephalophus callipygus,[],True,...,20.25,63125511.0,1,2334555,3487956,0.0,True,True,False,manager_verified
1,30853,Philantomba monticola,Blue Duiker,Mammalia,Artiodactyla,Bovidae,Philantomba,Philantomba monticola,[],True,...,120.25,372007915.0,1,2334567,3487968,0.0,False,True,False,labeller_verified
2,30853,Philantomba monticola,Blue Duiker,Mammalia,Artiodactyla,Bovidae,Philantomba,Philantomba monticola,[],True,...,120.25,371939333.0,1,2334527,3487928,0.0,False,True,False,labeller_verified
3,30853,Philantomba monticola,Blue Duiker,Mammalia,Artiodactyla,Bovidae,Philantomba,Philantomba monticola,[],True,...,120.25,372116737.0,1,2334531,3487932,0.0,False,True,False,labeller_verified
4,30853,Philantomba monticola,Blue Duiker,Mammalia,Artiodactyla,Bovidae,Philantomba,Philantomba monticola,[],True,...,120.25,371935659.0,1,2334533,3487934,0.0,False,True,False,labeller_verified


## 5. Unified Species Observations

Combine media, labels, and station metadata into flat "observation" records—one row per species identification. Each measurement type (camera, audio, eDNA) calls a different API, but all return consistent columns:

### 5.1 Camera Trap Observations (Image & Video)

`get_camera_trap_data()` returns one row per labelled camera segment, with columns:
- `project_system_record_id`, `device_id`, `latitude`, `longitude`: Station metadata
- `data_type`: `"image"` or `"video"`
- `measurement_type`: `"Camera"`
- `label`, `label_id`, `common_name`, `species`, `genus`, `family`, `order`: Species identification
- `media_file_record_id`, `segment_record_id`, `media_file_created_at`: File/segment IDs and timestamps
- `manager_verified`, `labeller_verified`: Verification status

In [18]:
df = get_camera_trap_data(hdr)
df.head()

,project_system_record_id,data_type,device_id,measurement_type,latitude,longitude,label_id,label,common_name,class_,...,duration_in_seconds,file_size,number_of_individuals,segment_record_id,label_record_id,prediction_accuracy,manager_verified,labeller_verified,blank,segment_verification_status
0,5863,image,00005,Camera,-0.078634,12.372708,30849,Cephalophus callipygus,Peters' Duiker,Mammalia,...,20.25,63125511.0,1,2334555,3487956,0.0,True,True,False,manager_verified
1,5863,image,00005,Camera,-0.078634,12.372708,30853,Philantomba monticola,Blue Duiker,Mammalia,...,120.25,372007915.0,1,2334567,3487968,0.0,False,True,False,labeller_verified
2,5863,image,00005,Camera,-0.078634,12.372708,30853,Philantomba monticola,Blue Duiker,Mammalia,...,120.25,371939333.0,1,2334527,3487928,0.0,False,True,False,labeller_verified
3,5863,image,00005,Camera,-0.078634,12.372708,30853,Philantomba monticola,Blue Duiker,Mammalia,...,120.25,372116737.0,1,2334531,3487932,0.0,False,True,False,labeller_verified
4,5863,image,00005,Camera,-0.078634,12.372708,30853,Philantomba monticola,Blue Duiker,Mammalia,...,120.25,371935659.0,1,2334533,3487934,0.0,False,True,False,labeller_verified


The below need implementing:

### 5.2 Audio Observations

`get_audio_observation_data()` returns one row per labelled audio segment, with columns:
- `project_system_record_id`, `device_id`, `latitude`, `longitude`: Station metadata
- `data_type`: `"audio"`
- `measurement_type`: `"Audio"`
- `label`, `label_id`, `common_name`, `species`, `genus`, `family`, `order`: Species identification
- `media_file_record_id`, `segment_record_id`, `media_file_created_at`: File/segment IDs and timestamps
- Additional `media_` and `segment_` columns containing full metadata from API responses

In [24]:
from naturecubepy import get_audio_observation_data

a_df = get_audio_observation_data(hdr)

ImportError: cannot import name 'get_audio_observation_data' from 'naturecubepy' (/Users/natimi/Projects/NatureCubePy/src/naturecubepy/__init__.py)

In [ ]:
stations = get_station_info(hdr, "audio")

In [ ]:
stations.measurement_type.unique()

<StringArray>
['Bioacoustic', 'Camera', 'eDNA']
Length: 3, dtype: str

In [ ]:
stations[stations['measurement_type'] == 'eDNA']

,system_type,feature_id,feature_name,system_name,device_id,project_system_record_id,record_count,measurement_type,data_type,project_system_record_start_timestamp,project_system_record_end_timestamp,geometry
9,Sample,4516,WC02,eDNA - Aquatic,DAR-2025-0631,6732,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.33868 0.01084)
64,Sample,4574,eDNA_AQ02,eDNA - Aquatic,DAR-2025-0632,6734,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.34514 -0.05563)
65,Sample,4575,eDNA_AQ03,eDNA - Aquatic,DAR-2025-0637,6737,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.1964 -0.14258)
66,Sample,4576,eDNA_AQ04,eDNA - Aquatic,DAR-2025-0640,6736,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.27904 -0.09633)
67,Sample,4577,eDNA_AQ05,eDNA - Aquatic,DAR-2025-0630,6740,0,eDNA,audio,2025-09-18 00:00:00+00:00,2025-10-04 00:00:00+00:00,POINT (12.42742 0.11861)
68,Sample,4578,eDNA_AQ06,eDNA - Aquatic,DAR-2025-0633,6731,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.33865 0.03011)
69,Sample,4579,eDNA_AQ07,eDNA - Aquatic,DAR-2025-0635,6735,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.33693 -0.09624)
70,Sample,4581,eDNA_AQ09,eDNA - Aquatic,DAR-2025-0647,6728,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.1954 -0.199)
71,Sample,4582,eDNA_AQ10,eDNA - Aquatic,DAR-2025-0726,6730,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.36335 -0.15984)
72,Sample,4583,eDNA_AQ11,eDNA - Aquatic,DAR-2025-0723,6738,0,eDNA,audio,2025-06-18 00:00:00+00:00,2025-07-03 00:00:00+00:00,POINT (12.20935 -0.19977)


### 5.3 eDNA Observations

`get_edna_observation_data()` returns one row per eDNA environmental DNA asset, with columns:
- `project_system_record_id`, `device_id`, `latitude`, `longitude`: Station metadata
- `data_type`: `"edna"`
- `measurement_type`: `"eDNA"`
- `label`, `label_id`, `common_name`, `species`, `genus`, `family`, `order`: Species identification (from API's species lists)
- `psr_id`: Primary species record ID from eDNA station
- Additional `media_` columns containing asset metadata (file names, timestamps, creation dates)

### 5.4 Unified Species Observations

`get_all_species_observations()` combines all three observation types (camera, audio, eDNA) into a single DataFrame with consistent columns. This is useful for cross-measurement-type analyses:
- Standardized columns: `project_system_record_id`, `device_id`, `data_type`, `measurement_type`, `latitude`, `longitude`, `label`, `label_id`, `common_name`, `species`, `genus`, `family`, `order`
- Each row represents a single species identification event, regardless of source
- `data_type` column (`"image"`, `"video"`, `"audio"`, `"edna"`) indicates measurement source
- `measurement_type` column (`"Camera"`, `"Audio"`, `"eDNA"`) indicates sensor type